# File function
This file will preprocess and extract neccessary info from `camma_mvor_2018.json` annotation file 

In [23]:
#importing neccessary packages and libraries
import json
from collections import defaultdict
import numpy as np

In [24]:
#loading in pose annotations 
try:
    with open('camma_mvor_2018.json') as f:
        data = json.load(f)
except FileNotFoundError:
    print("file not found")

except:
    print("other error")

annot_3D = data["annotations3D"] #3D pose annotations


#ensuring annotations where loaded properly
print(annot_3D[0]['keypoints3D'])

[630.6556928399972, -498.0674644841253, 2394.9087204313873, 2.0, 650.6029327608261, -435.673747489747, 2467.1504194692443, 2.0, 765.2336487731467, -415.1274711986172, 2567.2710725550846, 2.0, 471.51789534403446, -417.4315698597386, 2576.9883790848066, 2.0, 762.8635121989911, -103.45690191919803, 2816.927522526169, 2.0, 493.88975427043096, -88.01155362794417, 2806.6194635029433, 2.0, 739.0484619140625, -224.3656005859375, 2560.0, 2.0, 334.1165332364915, -217.05363208174617, 2561.7882492948274, 2.0, 737.6575317382812, -122.52092742919922, 2450.0, 2.0, 478.4024783369531, -167.64844196983597, 2436.253111187903, 2.0, 787.6575317382812, -72.52092742919922, 2500.0, 1.0, 541.8294915212192, -182.77253333431509, 2388.574626849282, 2.0]


## Frame by frame pose grouping 
Relevant information to extraction from each annotation
- `image_ids`
- `person_id`
- `keypoints3D`

Poses per frame will be stored in dictionary with the following shape: 

```python
    poses_per_frame = {
        frame_1: [ {pose1}, ... , {poseN}],
        ...
        frame_N: [ {pose1}, ... , {poseN}],
    }
```



In [25]:
poses_per_frame = defaultdict(list) 

for annot in annot_3D:
    frame = annot['image_ids']
    person = annot['person_id']
    keypoints = annot['keypoints3D']


    poses_per_frame[frame].append({
        'person': person,
        'keypoints': keypoints
    })

#inspect data to make sure it was parsed properly 
print(f"There's {len(poses_per_frame)} total frames") 
print(f"Frame 1: {list(poses_per_frame.values())[0]}")

multiPersonFrame = 0
for pose in list(poses_per_frame.values()):
    if len(pose)>1:
        multiPersonFrame +=1

print(f"Theres {multiPersonFrame} frames with more than one person")


There's 629 total frames
Frame 1: [{'person': 0, 'keypoints': [630.6556928399972, -498.0674644841253, 2394.9087204313873, 2.0, 650.6029327608261, -435.673747489747, 2467.1504194692443, 2.0, 765.2336487731467, -415.1274711986172, 2567.2710725550846, 2.0, 471.51789534403446, -417.4315698597386, 2576.9883790848066, 2.0, 762.8635121989911, -103.45690191919803, 2816.927522526169, 2.0, 493.88975427043096, -88.01155362794417, 2806.6194635029433, 2.0, 739.0484619140625, -224.3656005859375, 2560.0, 2.0, 334.1165332364915, -217.05363208174617, 2561.7882492948274, 2.0, 737.6575317382812, -122.52092742919922, 2450.0, 2.0, 478.4024783369531, -167.64844196983597, 2436.253111187903, 2.0, 787.6575317382812, -72.52092742919922, 2500.0, 1.0, 541.8294915212192, -182.77253333431509, 2388.574626849282, 2.0]}]
Theres 313 frames with more than one person


## Per operation grouping
Based on pose grouping above, frames will now be grouped per operation. Operations will be stored in a dictionary with the following shape:

```python
    operations = {
        day_x: {operation_x: [frames]}

    }
```

In [26]:
opRanges = { #frame span of operations
    'day1': {
        (0, 56)
    },
    'day2': {
        (0, 117),
        (118, 232),
        (233, 234),
        (235, 245),
        (246, 257),
        (258, 270),
        (271, 278),
        (279, 293),
        (294, 321),
        (322, 324),
        (325, 329)
    },
    'day3': {
        (0, 7),
        (8, 28),
        (29, 58),
        (59, 84),
        (85, 124),
        (125, 138),
        (139, 160),
        (161, 172),
        (173, 181),
        (182, 196),
        (197, 217)
    },
    'day4': {
        (0, 75),
        (76, 116),
        (117, 121)
    }
}

operations = defaultdict(lambda: defaultdict(list)) 
def getIndex(imageId): #extracts image index from id 
    return int(imageId.split('_')[0][-6:]) #ex: 10010000002_10020000002_10030000002 --> 000002 --> 2)

def getDay(imageId): #extracts day from id
    day = int(imageId.split('_')[0][0])

    match day:
        case 1:
            return 'day1'
        
        case 2:
            return 'day2'
        
        case 3:
            return 'day3'
        
        case 4:
            return 'day4'
    

#iterating through frames and mapping them to their respective operations 
for frameId, poses in poses_per_frame.items():
    dayString = getDay(frameId)
    frameIdx = getIndex(frameId)


    for opIdx, (opStart, opEnd) in enumerate(opRanges[dayString], start =1):
        
        if opStart <= frameIdx <= opEnd:
            opString = f"operation_{opIdx}"
            operations[dayString][opString].append({
                'frameIdx': frameIdx,
                'poses': poses 
            })
            break #make sure it only maps to one operation

#sort to put frames in order
for day in operations:
    for op in operations[day]:
        operations[day][op].sort(key=lambda f: f['frameIdx'])



### Inspecting data 

In [27]:
#Visualizing operation annotations
print(f"Day 1, operation 1: {operations['day1']['operation_1']}")

Day 1, operation 1: [{'frameIdx': 1, 'poses': [{'person': 0, 'keypoints': [-919.0739482366955, 220.82523959568664, 1500.285433128346, 2.0, -900.6188055769591, 269.0215963454077, 1591.8862113213054, 2.0, -767.3401648708726, 254.80329475774147, 1701.0095984835384, 2.0, -1062.8579020560446, 242.84780550162645, 1779.6725563248588, 2.0, -707.1937682188919, 473.3078621762493, 1983.514584502264, 1.0, -971.708986329872, 629.1873350132807, 1986.7901324341426, 2.0, -642.777364167334, 357.5466914290271, 1908.2174367030243, 1.0, -1093.1801297857166, 456.7583338835688, 1950.6948686629373, 2.0, -602.3968193964831, 574.8271083699042, 1919.106531333714, 1.0, -1117.1871539088602, 665.6120719028361, 2062.1952132462443, 2.0, -596.7772216796875, 643.4946899414062, 1883.0, 1.0, -1154.817825383358, 691.7864954734459, 2068.0941845246466, 2.0]}]}, {'frameIdx': 3, 'poses': [{'person': 1, 'keypoints': [-227.58989221610074, -184.0425039183756, 2313.0511787435516, 2.0, -233.268458343472, -170.6410320503951, 2410.

In [20]:
#Seeing how many operations have under annotated frames 

under10 = 0
total = 0

for day in operations:
    for op in operations[day]:
        total+=1
        if len(operations[day][op])<10:
            under10+=1

print(f"{under10}/{total} operations have less than 10 frames")
    

9/26 operations have less than 10 frames
